# `Imports & Setup`



---


*الجزء ده مسؤول عن تجهيز كل المكتبات والملفات اللي المشروع هيشتغل بيها.*


---




*   `cv2` علشان نفتح الكاميرا ونعرض الفيديو ونرسم على الوجه.
*   `face_recognition` علشان نكتشف الوشوش ونعمل مقارنة بينهم.
*  `numpy` علشان العمليات الحسابية وتحديد أقرب وجه مطابق.
*   `pickle` علشان نحفظ بيانات الوجوه بشكل دائم.
*   `os` علشان نتأكد إذا الملفات موجودة أو لا.
*   `datetime` علشان نسجل وقت وتاريخ الحضور.






In [1]:
import cv2
import face_recognition
import numpy as np
import pickle
import os
from datetime import datetime
from scipy.spatial import distance as dist

# Files
encodings_file = "encodings.pkl"
attendance_file = "attendance.csv"

c:\Users\LoQ\AppData\Local\Programs\Python\Python310\lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# **` Load Saved Faces`**


---


*الجزء ده بيحمّل الوجوه اللي اتسجلت قبل كده من الملف.*


---
`لو الملف موجود:`

 و الاسماء encoding يحمل الـ

 `لو مش موجود:`

 فاضية Lists يبدأ

In [2]:
if os.path.exists(encodings_file):

    with open(encodings_file, "rb") as f:
        data = pickle.load(f)

    known_encodings = data["encodings"]
    known_names = data["names"]

    print("Saved faces loaded successfully")

else:
    known_encodings = []
    known_names = []

    print("No saved faces found")

No saved faces found


# **` Create Attendance File`**


---

*الجزء ده بيعمل ملف الحضور لو مش موجود.*

*وده يعتبر قاعدة بيانات بسيطة للحضور والانصراف.*

---
`الملف بيتخزن فيه:`


*   الاسم
*   التاريخ
*   الوقت





In [3]:
if not os.path.exists(attendance_file):

    with open(attendance_file, "w") as f:
        f.write("Name,Date,Time\n")

print("Attendance file ready")

Attendance file ready


# **` Attendance Function`**


---

*الفنكشن دي مسؤولة عن تسجيل الحضور تلقائيًا.*

*الفكرة منها منع تكرار تسجيل نفس الشخص أكتر من مرة في نفس اليوم*

---
`لما الشخص يتعرف:`


*  تجيب الوقت والتاريخ الحالي.
*   تفتح ملف الحضور.
*  تتأكد إن الشخص متسجلش قبل كده النهارده.
*  لو مش متسجل → تضيفه في الملف.







In [4]:
def mark_attendance(name):

    now = datetime.now()

    date = now.strftime("%Y-%m-%d")
    time = now.strftime("%H:%M:%S")

    with open(attendance_file, "r+") as f:

        lines = f.readlines()

        already_logged = False

        for line in lines:

            data = line.strip().split(",")

            if len(data) >= 2:

                saved_name = data[0]
                saved_date = data[1]

                # منع التكرار في نفس اليوم
                if saved_name == name and saved_date == date:
                    already_logged = True
                    break

        if not already_logged:

            f.write(f"{name},{date},{time}\n")

            print(f"{name} attendance recorded")

EAR

In [5]:
def calculate_ear(eye):
    # حساب المسافات الرأسية بين جفون العين
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    # حساب المسافة الأفقية بين طرفي العين
    C = dist.euclidean(eye[0], eye[3])
    # معادلة الـ EAR
    ear = (A + B) / (2.0 * C)
    return ear

# إعدادات كشف الرمش
EYE_AR_THRESH = 0.22  # الحد الأدنى لفتحة العين (لو قل عنها يبقى رمش)
BLINK_CONSEC_FRAMES = 3  # عدد الفريمات المتتالية للتأكد إنها رمشة مش غمزة عارضة
COUNTER = 0
TOTAL_BLINKS = 0

# `** Register New Face**`


---
*الجزء ده مسؤول عن تسجيل مستخدم جديد.*

*بدل ما نحفظ صورة عادية، بنحفظ مميزات الوجه الرقمية علشان التعرف يبقى أسرع وأدق.*

---

`خطواته:`


*   المستخدم يدخل اسمه
*   الكاميرا تفتح
*   لما يضغط `s`:
     

1.   النظام يلقط صورة للوجه
2.   `Face Encoding` يستخرج الـ
3.  في الملف `Encodin`g  يحفظ الاسم والـ


















In [6]:
name = input("Enter your name: ")

video = cv2.VideoCapture(0)

print("Press 's' to save your face")
print("Press 'q' to quit")

saved = False

while True:

    ret, frame = video.read()

    if not ret:
        break

    cv2.imshow("Register Face", frame)

    key = cv2.waitKey(10) & 0xFF

    # Save Face
    if key == ord('s') and not saved:

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        faces = face_recognition.face_locations(rgb)
        encodings = face_recognition.face_encodings(rgb, faces)

        if len(encodings) > 0:

            known_encodings.append(encodings[0])
            known_names.append(name)

            # Save Permanently
            with open(encodings_file, "wb") as f:

                pickle.dump({
                    "encodings": known_encodings,
                    "names": known_names
                }, f)

            print("Face Registered Successfully!")

            saved = True
            break

        else:
            print("No face detected")

    # Quit
    elif key == ord('q'):
        break

video.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

Press 's' to save your face
Press 'q' to quit
Face Registered Successfully!


-1

## **`Face Recognition + Attendance`**



---
*ده الجزء الأساسي في المشروع.*

---

`النظام هنا:`




1.  يفتح الكاميرا.
2.  يكتشف الوجوه الموجودة.
3. ` Encoding`  حول الوجه ل
4.   يقارن الوجه بالوجوه المحفوظة

`لو لقى تطابق:`


*   يظهر اسم الشخص.
*   يسجل حضوره.

`لو ملقاش:`


*  `Unknown` يظهر



---
`استخدمنا:`






*   `compare_faces()` للمقارنة

*   `face_distance()` لمعرفة أقرب وجه مطابق.

*   `argmin()` لاختيار أقل مسافة وبالتالي أفضل Match.









In [7]:
video = cv2.VideoCapture(0)
TOTAL_BLINKS = 0 # تصفير العداد عند البدء
COUNTER = 0

print("Face Recognition Started")
print("Please Blink to record attendance...")

while True:
    ret, frame = video.read()
    if not ret: break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # استخراج نقاط الوجه (Landmarks) عشان نحدد العين
    face_landmarks_list = face_recognition.face_landmarks(rgb)
    faces = face_recognition.face_locations(rgb)
    encodings = face_recognition.face_encodings(rgb, faces)

    for (encoding, face_loc, landmarks) in zip(encodings, faces, face_landmarks_list):
        # 1. كشف الرمش أولاً
        leftEye = landmarks['left_eye']
        rightEye = landmarks['right_eye']
        ear = (calculate_ear(leftEye) + calculate_ear(rightEye)) / 2.0

        if ear < EYE_AR_THRESH:
            COUNTER += 1
        else:
            if COUNTER >= BLINK_CONSEC_FRAMES:
                TOTAL_BLINKS += 1
            COUNTER = 0

        # 2. التعرف على الوش
        matches = face_recognition.compare_faces(known_encodings, encoding)
        face_distances = face_recognition.face_distance(known_encodings, encoding)
        name = "Unknown"

        if len(face_distances) > 0:
            best_match = np.argmin(face_distances)
            if matches[best_match]:
                name = known_names[best_match]
                
                # ميزة الأمان: لا يسجل حضور إلا بعد كشف رمشة واحدة على الأقل
                if TOTAL_BLINKS >= 1:
                    mark_attendance(name)
                    status_color = (0, 255, 0) # أخضر لو رمش وسجل
                else:
                    status_color = (0, 165, 255) # برتقالي لو لسه مرمش
        
        # رسم المربع والبيانات
        top, right, bottom, left = face_loc
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, f"{name} | Blinks: {TOTAL_BLINKS}", (left, top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Secure Attendance System", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

video.release()
cv2.destroyAllWindows()

Face Recognition Started
Please Blink to record attendance...
